In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
df=pd.read_csv(r"C:\Users\GR0012AU\Downloads\covid_toy.csv")

In [3]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [5]:
df["fever"]=df["fever"].fillna(np.mean(df["fever"]))

In [6]:
df["fever"].isnull().sum()

0

In [8]:
df.isnull().sum()

age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64

In [12]:
independent_column=df.iloc[:,:5]
dependent_column=df[["has_covid"]]

In [13]:
independent_column

,age,gender,fever,cough,city
0,60,Male,103.0,Mild,Kolkata
1,27,Male,100.0,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
3,31,Female,98.0,Mild,Kolkata
4,65,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore
96,51,Female,101.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore
98,5,Female,98.0,Strong,Mumbai


In [14]:
dependent_column

,has_covid
0,No
1,Yes
2,No
3,No
4,No
...,...
95,No
96,Yes
97,No
98,No


In [16]:
independent_column["cough"].unique()

array(['Mild', 'Strong'], dtype=object)

# Aam Zindagi

In [5]:
from sklearn.preprocessing import OrdinalEncoder

In [18]:
OE=OrdinalEncoder(categories=[['Mild', 'Strong']])

In [30]:
df1=df.iloc[:,[3]]

In [31]:
df1.shape

(100, 1)

In [34]:
df1=pd.DataFrame(OE.fit_transform(df1),columns=["cough"])

In [35]:
df1

,cough
0,0.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
95,0.0
96,1.0
97,0.0
98,1.0


In [37]:
independent_column=independent_column.drop(["cough"],axis=1)

In [39]:
independent_column["cough"]=df1

In [40]:
independent_column

,age,gender,fever,city,cough
0,60,Male,103.0,Kolkata,0.0
1,27,Male,100.0,Delhi,0.0
2,42,Male,101.0,Delhi,0.0
3,31,Female,98.0,Kolkata,0.0
4,65,Female,101.0,Mumbai,0.0
...,...,...,...,...,...
95,12,Female,104.0,Bangalore,0.0
96,51,Female,101.0,Kolkata,1.0
97,20,Female,101.0,Bangalore,0.0
98,5,Female,98.0,Mumbai,1.0


In [41]:
df2=df[["gender","city"]]

In [42]:
df2

,gender,city
0,Male,Kolkata
1,Male,Delhi
2,Male,Delhi
3,Female,Kolkata
4,Female,Mumbai
...,...,...
95,Female,Bangalore
96,Female,Kolkata
97,Female,Bangalore
98,Female,Mumbai


In [6]:
from sklearn.preprocessing import OneHotEncoder

In [45]:
OHE=OneHotEncoder(drop="first",sparse=False, dtype=np.int32)

In [49]:
df2=pd.DataFrame(OHE.fit_transform(df2),columns=pd.get_dummies(df2,drop_first=True).columns)

In [50]:
df2

,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
0,1,0,1,0
1,1,1,0,0
2,1,1,0,0
3,0,0,1,0
4,0,0,0,1
...,...,...,...,...
95,0,0,0,0
96,0,0,1,0
97,0,0,0,0
98,0,0,0,1


In [56]:
independent_columns=pd.concat([independent_column,df2],axis=1)

In [60]:
independent_columns=independent_columns.drop(["gender","city"],axis=1)

In [61]:
independent_columns

,age,fever,cough,gender_Male,city_Delhi,city_Kolkata,city_Mumbai
0,60,103.0,0.0,1,0,1,0
1,27,100.0,0.0,1,1,0,0
2,42,101.0,0.0,1,1,0,0
3,31,98.0,0.0,0,0,1,0
4,65,101.0,0.0,0,0,0,1
...,...,...,...,...,...,...,...
95,12,104.0,0.0,0,0,0,0
96,51,101.0,1.0,0,0,1,0
97,20,101.0,0.0,0,0,0,0
98,5,98.0,1.0,0,0,0,1


In [7]:
from sklearn.preprocessing import LabelEncoder

In [70]:
LB=LabelEncoder()
dependent_columns=pd.DataFrame(LB.fit_transform(dependent_column),columns=dependent_column.columns)

In [71]:
dependent_columns

,has_covid
0,0
1,1
2,0
3,0
4,0
...,...
95,0
96,1
97,0
98,0


# Mentos Zindagi

In [8]:
from sklearn.compose import ColumnTransformer

In [9]:
from sklearn.impute import SimpleImputer

In [45]:
Transformer=ColumnTransformer(transformers=[
    ("tnf1",SimpleImputer(),["fever"]),
    ("tnf2",OrdinalEncoder(categories=[['Mild', 'Strong']]),["cough"]),
    ("tnf3",OneHotEncoder(sparse=False,drop='first'),['gender','city'])
],remainder="passthrough")

In [52]:
# transformer=ColumnTransformer(transformers=[("tnf4",LabelEncoder(),["has_covid"])],remainder="passthrough")
# LabelEncoder does not work with ColumnTransformer

In [41]:
from sklearn.model_selection import train_test_split

In [42]:
# x_train,x_test,y_train,y_test=train_test_split(independent_column,dependent_column,test_size=0.3,random_state=0)
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [43]:
x_train

,age,gender,fever,cough,city
60,24,Female,102.0,Strong,Bangalore
80,14,Female,99.0,Mild,Mumbai
90,59,Female,99.0,Strong,Delhi
68,54,Female,104.0,Strong,Kolkata
51,11,Female,100.0,Strong,Kolkata
...,...,...,...,...,...
96,51,Female,101.0,Strong,Kolkata
67,65,Male,99.0,Mild,Bangalore
64,42,Male,104.0,Mild,Mumbai
47,18,Female,104.0,Mild,Bangalore


In [46]:
Transformer.fit_transform(x_train).shape

(70, 7)

In [47]:
Transformer.fit_transform(x_test).shape

(30, 7)